# Agente para resolver Sudoku
**Mileidis Rojas y Gabriela Fonseca**  
Representación y Solución Automática de Problemas

El notebook muestra cómo se crea el entorno, cómo funcionan sus métodos y cómo se aplican Backtracking y Dijkstra en el Sudoku clásico y diagonal.

In [1]:
from pathlib import Path
import sys

# Buscar la raíz del proyecto desde el directorio actual
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "environment" / "sudoku.py").is_file()
     and (p / "data" / "instances.json").is_file()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Abrir dentro de la carpeta del proyecto.")
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from environment.sudoku import SudokuProblem
from environment.constraints import validate_complete_board
from instances.catalog import get_puzzle, clue_count
from instances.generator import generate_puzzle, count_solutions
from algorithms.backtracking import solve_backtracking
from algorithms.dijkstra import solve_dijkstra
from evaluation.comparison import compare_algorithms
from evaluation.metrics import result_to_row


def mostrar_tablero(tablero):
    display(pd.DataFrame(tablero).replace(0, "·"))

## 1. Creación de una instancia
El tablero es una matriz de 9 × 9. El cero representa una celda vacía. Las pistas iniciales no se pueden modificar.

Se usa un tablero fijo del catálogo para repetir las pruebas con la misma entrada. Los niveles fácil, medio y difícil tienen 48, 40 y 34 pistas, respectivamente.

In [2]:
tablero = get_puzzle("clasico", "Dificil")
problema = SudokuProblem(tablero, variante="clasico")

print("Variante:", problema.variante)
print("Pistas:", clue_count(problema.estado_inicial))
print("Celdas vacías:", problema.cantidad_vacias())
print("Conflictos iniciales:", problema.validar_estado_inicial())
print("Estado inicial:")
mostrar_tablero(problema.estado_inicial)

print("Celdas fijas (1 = pista, 0 = modificable):")
display(pd.DataFrame(problema.valores_iniciales).astype(int))

Variante: clasico
Pistas: 34
Celdas vacías: 47
Conflictos iniciales: []
Estado inicial:


,0,1,2,3,4,5,6,7,8
0,·,·,6,5,·,·,·,1,·
1,·,9,5,4,6,·,·,8,2
2,·,3,8,1,7,9,·,6,4
3,·,·,·,·,2,·,8,·,1
4,·,·,·,·,5,4,2,·,7
5,5,·,·,·,·,·,·,4,·
6,·,·,·,3,9,·,4,2,·
7,9,·,·,·,·,8,·,·,5
8,·,·,3,·,·,7,·,·,·


Celdas fijas (1 = pista, 0 = modificable):


,0,1,2,3,4,5,6,7,8
0,0,0,1,1,0,0,0,1,0
1,0,1,1,1,1,0,0,1,1
2,0,1,1,1,1,1,0,1,1
3,0,0,0,0,1,0,1,0,1
4,0,0,0,0,1,1,1,0,1
5,1,0,0,0,0,0,0,1,0
6,0,0,0,1,1,0,1,1,0
7,1,0,0,0,0,1,0,0,1
8,0,0,1,0,0,1,0,0,0


## 2. Métodos del entorno
Una acción es una tupla `(fila, columna, valor)`, con índices desde cero. MRV selecciona la celda vacía con menos candidatos. Solo se generan acciones válidas para esa celda.

La transición crea un estado nuevo sin cambiar el anterior. Cada asignación tiene costo 1. No se usa una función de recompensa en la búsqueda.

In [3]:
estado = problema.estado_inicial
acciones = problema.acciones_aplicables(estado)
print("Acciones aplicables:", acciones)
print("Código del estado:", problema.codigo(estado))
print("¿Estado inicial completo y válido?", problema.test_objetivo(estado))

if acciones:
    accion = acciones[0]
    fila, columna, valor = accion
    print("Candidatos de la celda:", problema.candidatos(estado, fila, columna))
    siguiente = problema.transicion(estado, accion)
    print("Costo:", problema.costo(estado, accion))
    print("Celdas vacías después de la acción:", problema.cantidad_vacias(siguiente))
    print("Conflictos del nuevo estado:", problema.validar_tablero(siguiente))
    print("¿Nuevo estado objetivo?", problema.test_objetivo(siguiente))
    assert estado[fila][columna] == 0
    assert siguiente[fila][columna] == valor
    assert problema.cantidad_vacias(siguiente) == problema.cantidad_vacias(estado) - 1
    mostrar_tablero(siguiente)

copia = problema.copiar()
assert copia.estado_inicial == problema.estado_inicial and copia is not problema

Acciones aplicables: [(1, 5, 3)]
Código del estado: 006500010095460082038179064000020801000054207500000040000390420900008005003007000
¿Estado inicial completo y válido? False
Candidatos de la celda: [3]
Costo: 1
Celdas vacías después de la acción: 46
Conflictos del nuevo estado: []
¿Nuevo estado objetivo? False


,0,1,2,3,4,5,6,7,8
0,·,·,6,5,·,·,·,1,·
1,·,9,5,4,6,3,·,8,2
2,·,3,8,1,7,9,·,6,4
3,·,·,·,·,2,·,8,·,1
4,·,·,·,·,5,4,2,·,7
5,5,·,·,·,·,·,·,4,·
6,·,·,·,3,9,·,4,2,·
7,9,·,·,·,·,8,·,·,5
8,·,·,3,·,·,7,·,·,·


## 3. Generación de tableros aleatorios
El generador construye una solución completa desde cero y elimina pistas sin perder la solución única. Se usan dos semillas distintas para mostrar tableros diferentes de forma reproducible. Sin semilla, se generan instancias aleatorias.

In [4]:
for variante in ("clasico", "diagonal"):
    primero = generate_puzzle(variante, "Facil", seed=42)
    segundo = generate_puzzle(variante, "Facil", seed=43)
    assert primero != segundo
    print("Variante:", variante)
    for numero, generado in enumerate((primero, segundo), start=1):
        instancia = SudokuProblem(generado, variante)
        soluciones = count_solutions(generado, variante, limit=2)
        assert not instancia.validar_estado_inicial()
        assert clue_count(generado) == 48
        assert soluciones == 1
        print(f"Tablero {numero}: {clue_count(generado)} pistas, {soluciones} solución")
        mostrar_tablero(generado)

Variante: clasico
Tablero 1: 48 pistas, 1 solución


,0,1,2,3,4,5,6,7,8
0,8,·,6,2,7,·,4,1,3
1,9,·,·,4,5,·,7,·,·
2,·,·,·,8,1,·,·,6,5
3,7,4,3,5,2,·,·,9,1
4,6,·,9,3,·,·,2,5,8
5,·,2,8,·,6,1,·,7,4
6,·,6,5,7,·,·,·,·,9
7,·,8,7,·,9,·,5,2,·
8,1,·,·,6,3,5,·,·,7


Tablero 2: 48 pistas, 1 solución


,0,1,2,3,4,5,6,7,8
0,·,4,·,9,1,8,6,·,·
1,5,2,·,7,·,3,·,1,8
2,·,8,·,5,·,·,·,·,3
3,4,9,8,·,5,·,3,·,·
4,7,·,3,4,9,1,·,·,5
5,·,1,5,·,8,·,4,9,·
6,6,7,4,·,·,5,2,8,9
7,8,·,·,6,2,·,·,7,4
8,9,·,2,8,7,·,1,·,6


Variante: diagonal
Tablero 1: 48 pistas, 1 solución


,0,1,2,3,4,5,6,7,8
0,3,1,5,2,7,·,·,·,6
1,·,·,·,·,5,6,1,·,9
2,·,·,·,8,·,·,·,5,·
3,1,9,·,5,2,8,6,7,4
4,8,6,·,3,4,·,·,9,·
5,5,·,4,9,·,·,3,·,8
6,4,3,1,7,8,2,9,6,5
7,·,5,7,1,9,4,2,8,·
8,·,·,·,·,3,·,4,·,·


Tablero 2: 48 pistas, 1 solución


,0,1,2,3,4,5,6,7,8
0,5,1,·,·,4,2,7,·,·
1,9,·,2,·,1,8,·,·,3
2,8,6,4,·,·,·,·,1,·
3,4,9,·,2,5,·,3,6,1
4,6,·,·,4,9,1,8,2,·
5,2,5,·,·,·,6,4,·,7
6,·,4,·,8,6,9,·,7,·
7,7,·,·,1,·,5,6,·,·
8,1,2,6,7,·,4,·,5,8


## 4. Backtracking
Explora una rama en profundidad. Si no puede continuar, retrocede y prueba otra asignación. Se usa el Sudoku clásico difícil del catálogo.

In [5]:
resultado_bt = solve_backtracking(
    problema.copiar(), max_expansions=250_000,
    time_limit_seconds=8, capture_events=False
)


def comprobar_solucion(resultado, instancia):
    assert resultado.status == "solved", resultado.message
    assert resultado.solution is not None
    assert not validate_complete_board(resultado.solution, instancia.variante)
    assert instancia.test_objetivo(resultado.solution)
    assert all(
        resultado.solution[f][c] == instancia.estado_inicial[f][c]
        for f in range(9) for c in range(9)
        if instancia.valores_iniciales[f][c]
    )
    assert resultado.metrics["solution_cost"] == instancia.cantidad_vacias()
    assert resultado.metrics["solution_depth"] == instancia.cantidad_vacias()

comprobar_solucion(resultado_bt, problema)
print(resultado_bt.message)
mostrar_tablero(resultado_bt.solution)
display(pd.DataFrame([result_to_row(resultado_bt)]))

Solución encontrada mediante búsqueda en profundidad con retroceso.


,0,1,2,3,4,5,6,7,8
0,4,7,6,5,8,2,9,1,3
1,1,9,5,4,6,3,7,8,2
2,2,3,8,1,7,9,5,6,4
3,3,4,7,9,2,6,8,5,1
4,6,1,9,8,5,4,2,3,7
5,5,8,2,7,3,1,6,4,9
6,7,6,1,3,9,5,4,2,8
7,9,2,4,6,1,8,3,7,5
8,8,5,3,2,4,7,1,9,6


,Algoritmo,Estado,Tiempo (s),Nodos generados,Nodos expandidos,Nodos visitados,Frontera máxima,Memoria aprox. (MB),Longitud solución,Costo solución,Retrocesos,Óptima
0,Backtracking,solved,0.041547,52,52,53,48,0.157,47,47.0,5,Sí


## 5. Dijkstra
Explora primero los estados con menor costo acumulado. Usa una cola de prioridad y controla los estados repetidos. Se aplica al mismo tablero que Backtracking.

In [6]:
resultado_d = solve_dijkstra(
    problema.copiar(), max_expansions=20_000,
    time_limit_seconds=8, capture_events=False
)
comprobar_solucion(resultado_d, problema)
print(resultado_d.message)
mostrar_tablero(resultado_d.solution)
display(pd.DataFrame([result_to_row(resultado_d)]))

Solución encontrada mediante Dijkstra (costo uniforme).


,0,1,2,3,4,5,6,7,8
0,4,7,6,5,8,2,9,1,3
1,1,9,5,4,6,3,7,8,2
2,2,3,8,1,7,9,5,6,4
3,3,4,7,9,2,6,8,5,1
4,6,1,9,8,5,4,2,3,7
5,5,8,2,7,3,1,6,4,9
6,7,6,1,3,9,5,4,2,8
7,9,2,4,6,1,8,3,7,5
8,8,5,3,2,4,7,1,9,6


,Algoritmo,Estado,Tiempo (s),Nodos generados,Nodos expandidos,Nodos visitados,Frontera máxima,Memoria aprox. (MB),Longitud solución,Costo solución,Retrocesos,Óptima
0,Dijkstra,solved,0.047301,76,76,77,4,0.0228,47,47.0,0,Sí


## 6. Sudoku diagonal
Además de las reglas del clásico, las dos diagonales principales deben contener los números del 1 al 9 sin repetición. Se usa la función de comparación del proyecto para aplicar ambos algoritmos al mismo tablero diagonal.

In [7]:
tablero_diagonal = get_puzzle("diagonal", "Dificil")
problema_diagonal = SudokuProblem(tablero_diagonal, "diagonal")
print("Estado inicial diagonal:")
mostrar_tablero(problema_diagonal.estado_inicial)

comparacion_diagonal, figura, resultados_diagonal, _ = compare_algorithms(
    tablero_diagonal, "diagonal"
)
for resultado in resultados_diagonal:
    comprobar_solucion(resultado, problema_diagonal)
    print(resultado.algorithm)
    mostrar_tablero(resultado.solution)

display(comparacion_diagonal)
figura.show(renderer="plotly_mimetype")

Estado inicial diagonal:


,0,1,2,3,4,5,6,7,8
0,·,8,7,6,·,·,9,1,4
1,·,1,·,·,·,·,·,6,2
2,·,·,9,·,8,·,5,·,·
3,·,7,8,·,·,·,·,·,3
4,·,·,·,·,·,·,7,8,1
5,·,3,6,·,·,·,·,5,9
6,7,·,·,3,9,5,·,2,·
7,·,·,·,8,·,1,3,·,5
8,·,·,·,·,·,4,1,9,·


Backtracking


,0,1,2,3,4,5,6,7,8
0,2,8,7,6,5,3,9,1,4
1,3,1,5,9,4,7,8,6,2
2,6,4,9,1,8,2,5,3,7
3,1,7,8,5,2,9,6,4,3
4,5,9,2,4,3,6,7,8,1
5,4,3,6,7,1,8,2,5,9
6,7,6,1,3,9,5,4,2,8
7,9,2,4,8,6,1,3,7,5
8,8,5,3,2,7,4,1,9,6


Dijkstra


,0,1,2,3,4,5,6,7,8
0,2,8,7,6,5,3,9,1,4
1,3,1,5,9,4,7,8,6,2
2,6,4,9,1,8,2,5,3,7
3,1,7,8,5,2,9,6,4,3
4,5,9,2,4,3,6,7,8,1
5,4,3,6,7,1,8,2,5,9
6,7,6,1,3,9,5,4,2,8
7,9,2,4,8,6,1,3,7,5
8,8,5,3,2,7,4,1,9,6


,Algoritmo,Estado,Tiempo (s),Nodos generados,Nodos expandidos,Nodos visitados,Frontera máxima,Memoria aprox. (MB),Longitud solución,Costo solución,Retrocesos,Óptima
0,Backtracking,solved,0.029269,47,47,48,48,0.1564,47,47.0,0,Sí
1,Dijkstra,solved,0.017634,47,47,48,1,0.0095,47,47.0,0,Sí


## 7. Comparación de los algoritmos
Se prueban las seis instancias del catálogo. Para cada tablero se ejecutan ambos algoritmos sin animación. Los tiempos corresponden a esta ejecución del notebook, no a las 36 pruebas del informe.

In [8]:
filas = []
for variante in ("clasico", "diagonal"):
    for nivel in ("Facil", "Medio", "Dificil"):
        entrada = get_puzzle(variante, nivel)
        instancia = SudokuProblem(entrada, variante)
        for resolver, limite in ((solve_backtracking, 250_000), (solve_dijkstra, 20_000)):
            resultado = resolver(instancia.copiar(), max_expansions=limite,
                                 time_limit_seconds=8, capture_events=False)
            comprobar_solucion(resultado, instancia)
            fila = result_to_row(resultado)
            fila["Tiempo (ms)"] = round(resultado.metrics["elapsed_seconds"] * 1000, 3)
            del fila["Tiempo (s)"]
            filas.append({"Variante": variante, "Nivel": nivel, **fila})

tabla = pd.DataFrame(filas)
display(tabla[["Variante", "Nivel", "Algoritmo", "Estado", "Tiempo (ms)",
               "Nodos expandidos", "Memoria aprox. (MB)", "Costo solución", "Óptima"]])
print("Soluciones comprobadas:", len(tabla), "de 12")

,Variante,Nivel,Algoritmo,Estado,Tiempo (ms),Nodos expandidos,Memoria aprox. (MB),Costo solución,Óptima
0,clasico,Facil,Backtracking,solved,19.913,33,0.1108,33.0,Sí
1,clasico,Facil,Dijkstra,solved,12.140,33,0.0077,33.0,Sí
2,clasico,Medio,Backtracking,solved,20.956,41,0.1368,41.0,Sí
3,clasico,Medio,Dijkstra,solved,20.882,41,0.0087,41.0,Sí
4,clasico,Dificil,Backtracking,solved,15.403,52,0.1570,47.0,Sí
5,clasico,Dificil,Dijkstra,solved,21.510,76,0.0228,47.0,Sí
6,diagonal,Facil,Backtracking,solved,10.375,33,0.1108,33.0,Sí
7,diagonal,Facil,Dijkstra,solved,10.735,33,0.0077,33.0,Sí
8,diagonal,Medio,Backtracking,solved,12.984,41,0.1368,41.0,Sí
9,diagonal,Medio,Dijkstra,solved,15.160,41,0.0087,41.0,Sí


Soluciones comprobadas: 12 de 12


## 8. Interpretación
Ambos algoritmos usan MRV y un costo de 1 por asignación. El costo final es igual al número de celdas vacías iniciales. Esto no implica que los dos exploren la misma cantidad de nodos.

La memoria es una estimación de las estructuras de búsqueda, no una medición directa de RAM. La cantidad de pistas no determina por sí sola la dificultad real del tablero.